# 🏋️ Notebook 03: Model Training & Evaluation

**Goal**: Train Model A (ML), Model B (DL), and Ensemble across 3 scenarios.

Checklist:
- [ ] 80/20 split — Model A, B, Ensemble
- [ ] 70/30 split — Model A, B, Ensemble
- [ ] 60/40 split — Model A, B, Ensemble
- [ ] RMSE, MAPE, MAE, R² per model per scenario
- [ ] Actual vs Predicted plots
- [ ] Rekapitulasi table

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.features.feature_engineering import TimeSeriesFeatureEngineer
from src.evaluation.metrics import ResultsTable, evaluate_all
from src.visualization.plots import (
    plot_actual_vs_predicted, plot_metrics_comparison,
    plot_heatmap, plot_results_table
)

# ── Model A (ML) ── change to your chosen model
from src.models.ml_models import XGBoostModel

# ── Model B (DL) ──
from src.models.dl_models import LSTMModel

# ── Ensemble ──
from src.models.ensemble_models import VotingEnsemble

print('✅ Imports OK')

In [ ]:
# ── Load & Configure ─────────────────────────────────────────────────
DATE_COL   = 'date'   # ← UPDATE
TARGET_COL = 'value'  # ← UPDATE

loader = DataLoader()
df = loader.load_csv('data/raw/train.csv')  # ← UPDATE
print(f'Dataset: {df.shape}')
df.head(3)

In [ ]:
# ── Results accumulator ──────────────────────────────────────────────
results = ResultsTable()

SCENARIOS = [
    ('80/20 Split', 0.80),
    ('70/30 Split', 0.70),
    ('60/40 Split', 0.60),
]

for scenario_name, train_ratio in SCENARIOS:
    print(f'\n{'='*60}')
    print(f'  SCENARIO: {scenario_name}')
    print(f'{'='*60}')

    # ── Temporal split ──────────────────────────────────────────────
    n = len(df)
    split = int(n * train_ratio)
    train_df = df.iloc[:split].copy()
    test_df  = df.iloc[split:].copy()

    # ── Feature engineering (leakage-safe) ─────────────────────────
    feat_eng = TimeSeriesFeatureEngineer(
        target_col=TARGET_COL, date_col=DATE_COL
    )
    train_fe = feat_eng.fit_transform(train_df).dropna()
    full_df  = pd.concat([train_df, test_df], ignore_index=True)
    test_fe  = feat_eng.transform(test_df, full_df).dropna()

    feat_cols = feat_eng.get_feature_columns(train_fe)
    X_train, y_train = train_fe[feat_cols], train_fe[TARGET_COL]
    X_test,  y_test  = test_fe[feat_cols],  test_fe[TARGET_COL]

    print(f'  Train: {len(X_train)} | Test: {len(X_test)} | Features: {len(feat_cols)}')

    # ────────────────────────────────────────────────────────────────
    # MODEL A: XGBoost (Machine Learning)
    # ────────────────────────────────────────────────────────────────
    print('\n  [MODEL A] XGBoost...')
    model_A = XGBoostModel(params={'n_estimators': 300, 'learning_rate': 0.05,
                                    'max_depth': 6, 'random_state': 42})
    model_A.fit_timed(X_train, y_train)
    pred_A  = model_A.predict_timed(X_test)
    results.add('XGBoost (A)', 'ML', scenario_name, y_test.values, pred_A,
                fit_time=model_A.fit_time_)

    plot_actual_vs_predicted(
        train_df[DATE_COL], y_train.values,
        test_df[DATE_COL], y_test.values, pred_A,
        model_name='XGBoost', scenario=scenario_name,
        save_path=f'results/plots/model_A_{scenario_name.replace("/","_")}.png'
    )
    plt.show()

    # ────────────────────────────────────────────────────────────────
    # MODEL B: LSTM (Deep Learning)
    # ────────────────────────────────────────────────────────────────
    print('\n  [MODEL B] LSTM...')
    model_B = LSTMModel(params={'units': [128, 64], 'dropout': 0.2,
                                 'epochs': 100, 'batch_size': 32,
                                 'lookback': 30, 'learning_rate': 0.001,
                                 'patience': 15})
    model_B.fit_timed(X_train, y_train)
    pred_B  = model_B.predict_timed(X_test)
    results.add('LSTM (B)', 'DL', scenario_name, y_test.values, pred_B,
                fit_time=model_B.fit_time_)

    plot_actual_vs_predicted(
        train_df[DATE_COL], y_train.values,
        test_df[DATE_COL], y_test.values, pred_B,
        model_name='LSTM', scenario=scenario_name,
        save_path=f'results/plots/model_B_{scenario_name.replace("/","_")}.png'
    )
    plt.show()

    # ────────────────────────────────────────────────────────────────
    # ENSEMBLE: Voting (XGBoost C + LSTM D)
    # ────────────────────────────────────────────────────────────────
    print('\n  [ENSEMBLE] VotingEnsemble (XGBoost + LSTM)...')
    from src.models.ml_models import XGBoostModel as XGB2
    from src.models.dl_models import LSTMModel as LSTM2
    model_C = XGB2(params={'n_estimators': 300, 'learning_rate': 0.05,
                            'max_depth': 6, 'random_state': 42})
    model_D = LSTM2(params={'units': [128, 64], 'dropout': 0.2,
                             'epochs': 100, 'batch_size': 32,
                             'lookback': 30, 'learning_rate': 0.001, 'patience': 15})
    ensemble = VotingEnsemble([model_C, model_D], weights=[0.5, 0.5])
    ensemble.fit_timed(X_train, y_train)
    pred_E  = ensemble.predict_timed(X_test)
    results.add('Voting (XGB+LSTM)', 'Ensemble', scenario_name, y_test.values, pred_E,
                fit_time=ensemble.fit_time_)

    plot_actual_vs_predicted(
        train_df[DATE_COL], y_train.values,
        test_df[DATE_COL], y_test.values, pred_E,
        model_name='VotingEnsemble', scenario=scenario_name,
        save_path=f'results/plots/ensemble_{scenario_name.replace("/","_")}.png'
    )
    plt.show()

print('\n✅ All scenarios complete!')

In [ ]:
# ── Rekapitulasi Results ──────────────────────────────────────────────
results_df = results.to_dataframe()
results.save('results/metrics/results_summary.csv')
results.print_summary()
results_df

In [ ]:
# ── Heatmaps ──────────────────────────────────────────────────────────
from src.visualization.plots import plot_heatmap

for metric in ['RMSE', 'MAPE (%)', 'MAE', 'R²']:
    fig = plot_heatmap(results_df, metric=metric,
                       save_path=f'results/plots/heatmap_{metric}.png')
    plt.show()

In [ ]:
# ── Best Model Per Scenario ───────────────────────────────────────────
best = results_df.loc[results_df.groupby('Scenario')['RMSE'].idxmin()]
print('\n🏆 Best Models (lowest RMSE):')
print(best[['Scenario', 'Model', 'RMSE', 'MAPE (%)', 'MAE', 'R²']].to_string(index=False))